# Courses for Roadmap generation

In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
# Fuction for filtering out non-Russian and non-English text
def contains_only_ru_en(text):

    text = str(text)
    pattern = r'^[A-Za-zА-Яа-яЁё0-9\s\W]+$'

    return bool(re.match(pattern, text))

In [3]:
# Large couse base
udemy = pd.read_csv('../data/raw/Course_info.csv')
# Russin language courses
stepik = pd.read_csv('../data/raw/stepik_courses_data.csv')
udemy_ru = pd.read_csv('../data/raw/udemy_courses_data.csv')
# Cousers, edx, udacity
edx = pd.read_csv('../data/raw/final_cleaned_dataset.csv')

In [4]:
# Filtering by category for IT courses
udemy_filtered = udemy[(udemy['category'].isin(['Development','IT & Software']))]
# Filter roadmap categories and ignore irrelevant ones
relevant_categories = [
    'Data Science',
    'Programming Languages',
    'Web Development',
    'Database Design & Development',
    'Network & Security',
    'Operating Systems & Servers',
    'Software Development Tools',
    'Software Engineering',
    'IT Certifications',
    'Other IT & Software',
]

udemy_filtered = udemy_filtered[udemy_filtered['subcategory'].isin(relevant_categories)]
udemy_filtered = udemy_filtered[udemy_filtered['language'].isin(['English','Russian'])]
print(f"Before filtering: {len(udemy)}")
print(f"After filtering: {len(udemy_filtered)}")

Before filtering: 209734
After filtering: 33887


In [5]:
# Filtering Stepik courses by language
stepik_filtered = stepik[stepik['title'].apply(contains_only_ru_en)]
print(f"Before filtering: {len(stepik)}")
print(f"After filtering: {len(stepik_filtered)}")

Before filtering: 2056
After filtering: 2029


In [6]:
# Filtering edx courses by language
edx_filtered = edx[edx['Course_Name'].apply(contains_only_ru_en)]
print(f"Before filtering: {len(edx)}")
print(f"After filtering: {len(edx_filtered)}")

Before filtering: 2377
After filtering: 2341


In [7]:
# Filtering udemy courses by language
udemy_ru_filtered = udemy_ru[udemy_ru['title'].apply(contains_only_ru_en)]
print(f"Before filtering: {len(udemy_ru)}")
print(f"After filtering: {len(udemy_ru_filtered)}")

Before filtering: 3443
After filtering: 3436


In [8]:
# Normalization of difficaulty levels
difficulty_mapping = {
    'Начальный уровень': 0,
    'Средний уровень': 1,
    'Продвинутый уровень': 2
}
stepik_filtered['dificulty'] = stepik_filtered['difficulty'].map(difficulty_mapping)
udemy_ru_filtered['dificulty'] = udemy_ru_filtered['difficulty'].map(difficulty_mapping)

In [9]:
# Standardization of dataframes to a common format

def standardize_stepik(df):
    return pd.DataFrame({
        'title': df['title'],
        'description': df['description'],
        'rating': df['rating'],
        'number_of_reviews': df['reviews_count'],
        'course_url':None,
        'topic': df['course'],
        'platform': 'Stepik',
        'difficulty': df['difficulty'],
        'certificate': df['certificate_available'],
        'priority': 1 
    })

def standardize_new_udemy(df): 
    return pd.DataFrame({
        'title': df['title'],
        'description': df['description'],
        'rating': df['rating'],
        'number_of_reviews': df['reviews_count'],
        'course_url': None,
        'topic': df['course'],
        'platform': 'Udemy',
        'difficulty': df['difficulty'],
        'certificate': df['certificate_available'],
        'priority': 2
    })

def standardize_edx(df):
    return pd.DataFrame({
        'title': df['Course_Name'],
        'description': df['Skills'], 
        'rating': df['Rating'],
        'number_of_reviews': df['Number of students'],
        'course_url': None, 
        'topic': df['Course_Name'],  
        'platform': df['Platform'],
        'difficulty': df['Level'],
        'certificate': None,
        'priority': 3
    })

def standardize_udemy_original(df):
    return pd.DataFrame({
        'title': df['title'],
        'description': df['headline'],
        'rating': df['avg_rating'],
        'number_of_reviews': df['num_reviews'],
        'course_url': 'https://www.udemy.com' + df['course_url'],
        'topic': df['topic'],
        'platform': 'Udemy',
        'difficulty': None,
        'certificate': None,
        'priority': 4  
    })

all_courses = pd.concat([
    standardize_stepik(stepik_filtered),
    standardize_new_udemy(udemy_ru_filtered),
    standardize_edx(edx_filtered),
    standardize_udemy_original(udemy_filtered),
], ignore_index=True)

print(f"Total courses: {len(all_courses)}")
print(all_courses['platform'].value_counts())

Total courses: 41693
platform
Udemy       37323
Stepik       2029
Coursera     1193
EdX           900
Udacity       248
Name: count, dtype: int64


In [1]:
# Fuction that prioritizes russian language courses and relevance of the skill in the title, then by rating and number of reviews

def find_courses_by_skill(
    skill: str,
    all_courses: pd.DataFrame,
    top_n: int = 2
) -> list:
    
    skill_clean = skill.lower().replace('_', ' ')
    results = []
    used_titles = set() 

    for priority in sorted(all_courses['priority'].unique()):
        if len(results) >= top_n:
            break

        platform_df = all_courses[all_courses['priority'] == priority]

        level1 = platform_df[
            platform_df['title'].str.lower().str[:40]
            .str.contains(skill_clean, na=False)
        ].copy()
        level1['relevance'] = level1['title'].str.lower().str.find(skill_clean)
        level1 = level1.sort_values(['relevance', 'rating','number_of_reviews'], ascending=[True, False, False])

        level2 = platform_df[
            (~platform_df.index.isin(level1.index)) &
            platform_df['title'].str.lower().str.contains(skill_clean, na=False)
        ].sort_values(['rating', 'number_of_reviews'], ascending=False)

        level3 = platform_df[
            (~platform_df.index.isin(level1.index)) &
            (~platform_df.index.isin(level2.index)) &
            platform_df['description'].str.lower().str.contains(skill_clean, na=False)
        ].sort_values(['rating', 'number_of_reviews'], ascending=False)

        combined = pd.concat([level1, level2, level3])

        for _, row in combined.iterrows():
            if len(results) >= top_n:
                break
            if row['title'] not in used_titles:
                results.append({
                    'title': row['title'],
                    'description': row['description'],
                    'rating': row.get('rating'),
                    'number_of_reviews': row.get('number_of_reviews'),
                    'course_url': row.get('course_url'),
                    'difficulty': row.get('difficulty'),
                    'certificate': row.get('certificate'),
                    'platform': row['platform']
                })
                used_titles.add(row['title'])

    return results

In [11]:
find_courses_by_skill('git', all_courses, top_n=5)

[{'title': 'GitLab CI/CD: Автоматизация DevOps и деплой нейросети — Stepik',
  'description': 'Освоим GitLab CI за 54 урока, научимся писать CI/CD пайплайны: от основ до продвинутых фишек — шаблонизация, оптимизация, кэширование. Напишем пайплайн для нейросети. Подготовимся к собесам и разберём сложные вопросы! 🔥',
  'rating': 5.0,
  'number_of_reviews': '5',
  'course_url': None,
  'difficulty': 'Начальный уровень',
  'certificate': 'False',
  'platform': 'Stepik 🇷🇺'},
 {'title': 'GitLab CI/CD: Автоматизация DevOps и деплой нейросети',
  'description': 'Освоим GitLab CI за 54 урока, научимся писать CI/CD пайплайны: от основ до продвинутых фишек — шаблонизация, оптимизация, кэширование. Напишем пайплайн для нейросети. Подготовимся к собесам и разберём сложные вопросы! 🔥',
  'rating': 5.0,
  'number_of_reviews': '5',
  'course_url': None,
  'difficulty': 'Начальный уровень',
  'certificate': 'True',
  'platform': 'Stepik 🇷🇺'},
 {'title': 'Git и Github – базовый курс для разработчика — S

In [ ]:
# Improved function that prioritizes russian language courses and relevance of the skill in the title, then by rating and number of reviews, with better handling of duplicates and missing values
def find_fair_courses_by_skill(
    skill: str,
    all_courses: pd.DataFrame,
    top_n: int = 5
) -> list:
    
    skill_clean = skill.lower().replace('_', ' ')
    df = all_courses.copy()
    diff_map = {0: 'Beginner', 1: 'Intermediate', 2: 'Advanced'}

   
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(0)
    df['number_of_reviews'] = pd.to_numeric(df['number_of_reviews'], errors='coerce').fillna(0)
    
    df['_title_lower'] = df['title'].str.lower()
    df['_desc_lower'] = df['description'].astype(str).str.lower()
    
    safe_skill = re.escape(skill_clean)

    pattern = rf'(?<!\w){safe_skill}(?!\w)'

    mask_in_title = df['_title_lower'].str.contains(pattern, regex=True, na=False)
    mask_in_desc = df['_desc_lower'].str.contains(pattern, regex=True, na=False)
    
    combined_mask = mask_in_title | mask_in_desc
    df = df[combined_mask].copy()
    
    if df.empty:
        return []

    mask_in_title = mask_in_title[combined_mask]
    mask_in_desc = mask_in_desc[combined_mask]

    mask_level_1 = df['_title_lower'].str[:40].str.contains(skill_clean, regex=False, na=False)
    mask_level_2 = mask_in_title & ~mask_level_1
    mask_level_3 = mask_in_desc & ~mask_in_title

    conditions = [mask_level_1, mask_level_2, mask_level_3]
    df['match_level'] = np.select(conditions, [1, 2, 3], default=4)

    df['fair_score'] = df['rating'] * np.log1p(df['number_of_reviews'])


    df = df.sort_values(
        by=['match_level', 'fair_score'],
        ascending=[True, False]
    )

    df = df.drop_duplicates(subset=['title'], keep='first')
    df['difficulty_label'] = df['difficulty'].map(diff_map)
    df['difficulty_label'] = df['difficulty_label'].replace({pd.NA: None, float('nan'): None})
    results = []
    for _, row in df.head(top_n).iterrows():
        results.append({
            'title': row['title'],
            'platform': row['platform'],
            'rating': round(row['rating'], 2),
            'reviews': int(row['number_of_reviews']),
            'difficulty': row.get('difficulty_label'),
            'certificate': row.get('certificate'),
            'course_url': row.get('course_url')
        })

    return results

In [13]:
find_fair_courses_by_skill('git', all_courses, top_n=5)

[{'title': 'IBM: Git and GitHub Basics',
  'platform': 'EdX',
  'rating': 4.67,
  'reviews': 158284,
  'difficulty': 'Beginner',
  'certificate': None,
  'course_url': None},
 {'title': 'Version Control with Git',
  'platform': 'Udacity',
  'rating': 4.58,
  'reviews': 94373,
  'difficulty': 'Beginner',
  'certificate': None,
  'course_url': None},
 {'title': 'Git Complete: The definitive, step-by-step guide to Git',
  'platform': 'Udemy 🇺🇸',
  'rating': 4.47,
  'reviews': 25413,
  'difficulty': None,
  'certificate': None,
  'course_url': 'https://www.udemy.com/course/git-complete/'},
 {'title': 'The Git & Github Bootcamp',
  'platform': 'Udemy 🇺🇸',
  'rating': 4.78,
  'reviews': 11673,
  'difficulty': None,
  'certificate': None,
  'course_url': 'https://www.udemy.com/course/git-and-github-bootcamp/'},
 {'title': 'Git Going Fast: One Hour Git Crash Course',
  'platform': 'Udemy 🇺🇸',
  'rating': 4.4,
  'reviews': 16321,
  'difficulty': None,
  'certificate': None,
  'course_url': 'htt

In [ ]:
# Function to get roadmap with courses for each skill
def get_roadmap_with_courses(
    roadmap: dict,
    all_courses: pd.DataFrame,
    top_n: int = 2
) -> dict:
    result = {}
    for category, skills in roadmap.items():
        result[category] = {}
        for skill in skills:
            courses = find_fair_courses_by_skill(
                skill, all_courses, top_n=top_n
            )
            result[category][skill] = {'courses': courses}
    return result

In [20]:
roadmap_with_courses = get_roadmap_with_courses({'cloud': ['aws', 'azure'],
 'databases': ['sql server', 'mysql'],
 'libraries': ['spark', 'tensorflow'],
 'programming': ['python', 'sql'],
 'other': ['git', 'docker']}, all_courses, top_n=1)

In [21]:
roadmap_with_courses

{'cloud': {'aws': {'courses': [{'title': 'AWS: Amazon DynamoDB: Building NoSQL Database-Driven Applications',
     'platform': 'EdX',
     'rating': 5.0,
     'reviews': 160715,
     'difficulty': 'Intermediate',
     'certificate': None,
     'course_url': None}]},
  'azure': {'courses': [{'title': 'Cloud Developer using Microsoft Azure',
     'platform': 'Udacity',
     'rating': 4.58,
     'reviews': 102130,
     'difficulty': 'Intermediate',
     'certificate': None,
     'course_url': None}]}},
 'databases': {'sql server': {'courses': [{'title': 'Mastering SQL Server 2016 Integration Services (SSIS)-Part 1',
     'platform': 'Udemy 🇺🇸',
     'rating': 4.52,
     'reviews': 5910,
     'difficulty': None,
     'certificate': None,
     'course_url': 'https://www.udemy.com/course/masteringssis2016/'}]},
  'mysql': {'courses': [{'title': 'The Ultimate MySQL Bootcamp: Go from SQL Beginner to Expert',
     'platform': 'Udemy 🇺🇸',
     'rating': 4.64,
     'reviews': 73492,
     'difficu